# Multiple Ways to Create DataFrames

| # | Method | Schema Control | Metadata | Best For |
|---|--------|----------------|----------|----------|
| 01 | Column Name List (Schema Inference) | None (inferred) | ❌ | Quick prototyping |
| 02 | DDL (SQL) String Schema | Type + nullable | ❌ | Most production use |
| 03 | StructType / StructField | Type + nullable + metadata | ✅ | Column-level documentation |
| 04 | Python Dictionaries | None (inferred) | ❌ | JSON-like data |
| 05 | PySpark Row Objects | None (inferred) | ❌ | Programmatic row building |
| 06 | Empty DataFrame | Type + nullable | ❌ | Initialization / fallback |
| 07 | `spark.range` | Fixed (`id: Long`) | ❌ | Synthetic / benchmark data |

> [!TIP]

> 💡 **Recommended method — 02. DDL (SQL) String Schema**
> Concise, readable, and type-safe with `NOT NULL` support. Use **03. StructType / StructField** when you need per-column `metadata` (descriptions, tags). Reserve **01. Column Name List** for quick prototyping only — inferred schemas default all columns to nullable and add inference overhead.

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("create-df-tutorial").getOrCreate()


## 01. Quick Prototyping — Column Name List with Schema Inference

- 💡 Easiest method: pass a list of column names and Spark **infers** data types from the values
- Best suited for quick exploration where schema precision is not critical
- Schema inference adds a small overhead — for production pipelines prefer an explicit schema (Methods 02 or 03)

> [!NOTE]
> `spark.sparkContext.parallelize(data).toDF(columns)` is an RDD-based alternative that is **not supported** on Serverless compute (Spark Connect).

In [0]:

# Method 01 — Schema inference from a column-name list
# Spark inspects the data to infer each column's type automatically.
# No explicit type declarations are needed, but all columns default to nullable.
data = [
    (101, "Alice", "Engineering", 95000.0, True),
    (102, "Bob", "Sales", 72000.5, True),
    (103, "Charlie", "Marketing", None, False)
]

columns = ["emp_id", "name", "department", "salary", "is_active"]

# createDataFrame() accepts a plain list of column names as the schema argument.
emp_inferred_df = spark.createDataFrame(data=data, schema=columns)

emp_inferred_df.printSchema()
emp_inferred_df.show(10)


## 02. DDL (SQL) String Schema — Quick Explicit Schema

- 💡 Recommended for most use cases: concise, readable, and type-safe
- Schema is defined as a DDL string using SQL type names (`INTEGER`, `STRING`, `DOUBLE`, etc.)
- Supports `NOT NULL` constraints — but does **not** support column-level metadata (see Method 03)

In [0]:

# Method 02 — DDL string schema (SQL-style column definitions)
# Each column is declared as "name TYPE [NOT NULL]" — Spark parses it into a StructType.
data = [
    (101, "Alice", "Engineering", 95000.0, True),
    (102, "Bob", "Sales", 72000.5, True),
    (103, "Charlie", "Marketing", None, False)
]

schema = """
    emp_id INTEGER NOT NULL,
    name STRING NOT NULL,
    department STRING,
    salary DOUBLE,
    is_active BOOLEAN
"""

emp_ddl_df = spark.createDataFrame(data=data, schema=schema)

emp_ddl_df.printSchema()
emp_ddl_df.show(10)


## 03. Native PySpark Data Types — StructType / StructField (Full Control)

- 💡 Recommended when you need **column-level metadata** or fine-grained nullable control
- Most verbose approach, but the only method that supports per-field `metadata` (e.g., descriptions, tags)
- Useful when schema documentation must travel with the data in catalog operations

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
    BooleanType,
)

def printSchemaWithMetadata(dataframe):
    """Prints a PySpark schema tree along with column-level metadata."""
    # Access .schema once before iterating to avoid repeated Analyze RPCs
    fields = dataframe.schema.fields
    for field in fields:
        meta_str = f" (Metadata: {field.metadata})" if field.metadata else ""
        null_str = "nullable = true" if field.nullable else "nullable = false"
        print(f" |-- {field.name}: {field.dataType.simpleString()} ({null_str}){meta_str}")


# Method 03 — StructType / StructField with per-field metadata
# Each StructField specifies: name, data type, nullable flag, and optional metadata dict.
# The metadata dict can hold descriptions, tags, or custom annotations.
data = [
    (101, "Alice", "Engineering", 95000.0, True),
    (102, "Bob", "Sales", 72000.5, True),
    (103, "Charlie", "Marketing", None, False)
]

schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True),
    StructField("is_active", BooleanType(), False, metadata={
        "description": "Indicates whether the employee is currently active in the organization",
    }),
])

emp_struct_df = spark.createDataFrame(data=data, schema=schema)

emp_struct_df.printSchema()
printSchemaWithMetadata(emp_struct_df)
emp_struct_df.show(10)


## 04. Using a List of Python Dictionaries

- 💡 Most natural approach for Python developers working with JSON-like data
- Spark infers the schema from dictionary keys and value types — no schema argument needed
- All dictionaries should have consistent keys; missing keys produce `null` values

In [0]:

# Method 04 — List of Python dictionaries (schema inferred from keys and values)
# Each dict becomes a row; keys become column names. No explicit schema required.
dict_data = [
    {"emp_id": 101, "name": "Alice", "dept": "Engineering"},
    {"emp_id": 102, "name": "Bob", "dept": "Sales"},
    {"emp_id": 102, "name": "Bob"},
]

df_dicts = spark.createDataFrame(dict_data)

df_dicts.printSchema()
df_dicts.show(10)


## 05. Using PySpark Row Objects

- 💡 Spark-native row construction with named fields — useful for programmatic row building
- Schema is inferred from the keyword arguments passed to each `Row(...)` call
- Supports complex types like lists and nested structs out of the box

In [0]:
from pyspark.sql.types import Row

# Method 05 — PySpark Row objects with named keyword arguments
# Each Row is constructed with explicit field names; Spark infers types from the values.
row_data = [
    Row(emp_id=101, name="Alice", skills=["Spark", "Python"]),
    Row(emp_id=102, name="Bob", skills=["SQL", "Scala"])
]

df_rows = spark.createDataFrame(row_data)

df_rows.printSchema()
df_rows.show(10)


## 06. Special Case — Empty DataFrame with Schema

- Creates a DataFrame with a defined schema but **zero rows**
- Useful for initializing tables, testing transformations, or returning fallback empty results from functions

In [0]:

# Special Case — Empty DataFrame with a DDL string schema and no data rows.
# The schema is enforced but show() returns an empty result set.
empty_df = spark.createDataFrame([], schema="id INT, name STRING, amount DOUBLE")

empty_df.printSchema()
empty_df.show()


## 07. Special Case — Sequential Number DataFrame (`spark.range`)

- Generates a single-column DataFrame (`id` of type `LongType`) with a numeric sequence
- Supports `start`, `end` (exclusive), `step`, and `numPartitions` parameters
- Ideal for generating synthetic or benchmark data without needing an external source

In [0]:

# Special Case — spark.range() generates a sequential LongType "id" column.
# Parameters: start, end (exclusive), step, numPartitions.
df_range = spark.range(start=1, end=10, step=2, numPartitions=4)

df_range.printSchema()
df_range.show()